# MLB Player Cluster Analysis v2

**Author:** Jordan Harris

## Purpose

While looking for new project ideas, I came across [this post](https://www.reddit.com/r/Sabermetrics/comments/1mob27g/finding_mlb_batter_types_using_kmeans_clustering/) which got me thinking. How should we categorize players? What factors set players apart from others aside from AVG and OPS? I decided to investigate these questions a little further by focusing on different areas than the redditor. Instead of analyzing expected stats, I chose to use actuals to avoid mischaracterizing a player due to preseason projections, and instead of using percentiles from BaseballSavant I decided to use the raw stats. Additionally, my analysis looks at players through multiple seasons in an attempt to see if a player's "style" changes throughout their career (Spoiler: some do!).

## Data

We use two files from the Lahman baseball database:

- **Batting.csv** — season-level batting counting stats by player
- **People.csv** — player biographical info, used here only for `nameFirst`/`nameLast`

Both files are expected in a `files/` folder next to this notebook.


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")  # roughly mirrors ggplot2::theme_light()
%matplotlib inline

DATA_DIR = "files"
OUTPUT_DIR = "output"
RANDOM_STATE = 42
MIN_AB = 300
YEAR_START, YEAR_END = 2000, 2023
K_RANGE = range(2, 9)   # 2..8 inclusive, for the elbow plot
K_FINAL = 3             # final number of clusters used

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
batting = pd.read_csv(os.path.join(DATA_DIR, "Batting.csv"))

# People.csv contains some non-UTF8 bytes in accented names (e.g. "Acuna"),
# so we read it with a fallback encoding (latin1) to avoid parse errors.
people = pd.read_csv(
    os.path.join(DATA_DIR, "People.csv"), encoding="latin1"
)[["playerID", "nameFirst", "nameLast"]]
people["playerName"] = people["nameFirst"] + " " + people["nameLast"]

batting.head()


### Data manipulation

We cluster on single-season stats rather than career totals, this is to see if players evolve throughout their careers. We also restrict to **2000-2023** (the "modern era") and require at least 300 AB in a season, to avoid noisy, small-sample seasons.


In [ ]:
season = (
    batting[(batting["yearID"] >= YEAR_START) & (batting["yearID"] <= YEAR_END)]
    .groupby(["playerID", "yearID"], as_index=False)
    .agg(
        AB=("AB", "sum"),
        H=("H", "sum"),
        X2B=("2B", "sum"),
        X3B=("3B", "sum"),
        HR=("HR", "sum"),
        BB=("BB", "sum"),
        SO=("SO", "sum"),
        HBP=("HBP", "sum"),
        SF=("SF", "sum"),
        SB=("SB", "sum"),
        CS=("CS", "sum"),
    )
)

season = season[season["AB"] >= MIN_AB].copy()

season["PA"] = season["AB"] + season["BB"] + season["HBP"] + season["SF"]
season["singles"] = season["H"] - season["X2B"] - season["X3B"] - season["HR"]
season["AVG"] = season["H"] / season["AB"]
season["OBP"] = (season["H"] + season["BB"] + season["HBP"]) / season["PA"]
season["SLG"] = (
    season["singles"] + 2 * season["X2B"] + 3 * season["X3B"] + 4 * season["HR"]
) / season["AB"]
season["ISO"] = season["SLG"] - season["AVG"]
season["BB_pct"] = season["BB"] / season["PA"]
season["K_pct"] = season["SO"] / season["PA"]
season["SB_rate"] = season["SB"] / season["PA"]  # speed/aggressiveness proxy

print(f"Qualified player-seasons ({YEAR_START}-{YEAR_END}, AB >= {MIN_AB}): {len(season)}")
season.head()


## Feature Selection

We deliberately cluster on **rate stats**, not counting stats. Clustering on raw HR or SB totals would mostly separate players by playing time (even with min 300 PA, there will be discrepancies between starters and other role players), not by style. AVG, OBP, ISO, BB%, K%, and SB rate describe how a player produces value offensively, independent of how many plate appearances they got.


In [ ]:
feature_cols = ["AVG", "OBP", "ISO", "BB_pct", "K_pct", "SB_rate"]
features = season[feature_cols]

scaler = StandardScaler()
X = scaler.fit_transform(features)  # standardization is essential for k-means


## 3. Choosing K

We evaluate cluster quality across a range of `k` using the elbow method (within-cluster sum of squares).


In [ ]:
wss = []
for k in K_RANGE:
    km_i = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE)
    km_i.fit(X)
    wss.append(km_i.inertia_)

k_results = pd.DataFrame({"k": list(K_RANGE), "wss": wss})

plt.figure(figsize=(8, 6))
plt.plot(k_results["k"], k_results["wss"], marker="o", markersize=6)
plt.title("Elbow Method")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Total within-cluster sum of squares")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "elbow_method.png"), dpi=150)
plt.show()


The elbow bends meaningfully around **k=5**, which also produces clusters with clear, interpretable baseball meaning. We proceed with k=3 for this version of the analysis.


## 4. K-Means Clustering


In [ ]:
km = KMeans(n_clusters=K_FINAL, n_init=25, random_state=RANDOM_STATE)
season["cluster"] = km.fit_predict(X) + 1  # 1-indexed, to mirror R's factor levels
season["cluster"] = season["cluster"].astype("category")

# Attach player names for readability
season = season.merge(people, on="playerID", how="left")
season.head()


### Cluster profiles


In [ ]:
centroids = (
    season.groupby("cluster", observed=True)
    .agg(
        n=("playerID", "size"),
        AVG=("AVG", "mean"),
        OBP=("OBP", "mean"),
        ISO=("ISO", "mean"),
        BB_pct=("BB_pct", "mean"),
        K_pct=("K_pct", "mean"),
        SB_rate=("SB_rate", "mean"),
    )
    .round(3)
    .reset_index()
    .sort_values("cluster")
)
centroids


Based on these centroids, the three clusters map onto recognizable hitter archetypes:

| Cluster profile | Archetype |
|---|---|
| 1. High AVG/OBP/ISO, high BB% | **Power hitters** |
| 2. High AVG, moderate power, low K% | **All Around Hitters** |
| 3. Below-average AVG/OBP, low power | **Contact Speedsters** |

### Example players per cluster (2023 season)


In [ ]:
sample_2023 = (
    season[season["yearID"] == 2023]
    .groupby("cluster", observed=True)
    .head(6)[["cluster", "playerName", "AVG", "OBP", "ISO", "K_pct", "SB_rate"]]
    .sort_values("cluster")
)
sample_2023


## Visualizing the Clusters

We use PCA to project the 6-dimensional feature space down to 2D for visualization.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pcs = pca.fit_transform(X)
season["PC1"] = pcs[:, 0]
season["PC2"] = pcs[:, 1]

plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=season,
    x="PC1",
    y="PC2",
    hue="cluster",
    alpha=0.5,
    palette="deep",
)
plt.gca().set_title(
    "MLB Hitter Archetypes (2000-2023, AB >= 300)\n"
    "K-means clustering on AVG, OBP, ISO, BB%, K%, SB rate",
    fontsize=13,
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="cluster")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_clusters.png"), dpi=150)
plt.show()


## Player Lookups

A useful application of this clustering is tracking how a specific player's archetype has shifted over their career — for example, young speed-first players often add power as they mature and "graduate" into the elite all-around cluster.


In [ ]:
def lookup_player(name):
    mask = season["playerName"].str.contains(name, case=False, na=False)
    cols = ["playerName", "yearID", "AVG", "OBP", "ISO", "K_pct", "SB_rate", "cluster"]
    return season.loc[mask, cols].sort_values("yearID")

lookup_player("Buxton")


In [ ]:
lookup_player("Betts")
